# 📈 Stock Trend Predictor (High Accuracy Version)
Improved with advanced features + XGBoost model.


In [ ]:
!pip install yfinance pandas numpy scikit-learn matplotlib xgboost

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

In [ ]:
df = yf.download('AAPL', start='2010-01-01', end='2024-01-01')
df.columns = df.columns.get_level_values(0)
df.head()

In [ ]:
# Feature Engineering
df['Return'] = df['Close'].pct_change()
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

for w in [3,5,10,20,50]:
    df[f'MA_{w}'] = df['Close'].rolling(w).mean()

df['Volatility'] = df['Return'].rolling(10).std()
df['Momentum'] = df['Close'] - df['Close'].shift(10)

for lag in range(1,6):
    df[f'Lag_{lag}'] = df['Close'].shift(lag)

df['Target'] = np.where(df['Close'].shift(-1) > df['Close'], 1, 0)

df.dropna(inplace=True)
print(df.shape)

In [ ]:
X = df.drop(['Target'], axis=1)
y = df['Target']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
importances = model.feature_importances_
features = X.columns

plt.figure(figsize=(8,6))
plt.barh(features, importances)
plt.title('Feature Importance')
plt.show()